<a href="https://colab.research.google.com/github/blskyue/AI-Research-Agent/blob/main/ai_research_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
from google.colab import userdata
import os

# سحب مفاتيح secrets
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

print("تم تفعيل المفاتيح بنجاح!")

تم تفعيل المفاتيح بنجاح!


In [42]:
!pip install -U langchain-openai langgraph langchain-community tavily-python

In [43]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
import requests
from langchain.tools import tool

# 1. تعريف الأدوات المطلوبة
internet_search = TavilySearchResults(k=3) # أداة البحث

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    try:
        response = requests.get(url, timeout=10.0)
        response.raise_for_status()
        return response.text
    except Exception as e:
        return f"Error fetching URL: {e}"

tools = [internet_search, fetch_url]

# 2. إعداد الموديل (Nemotron)
model_nemotron = ChatOpenAI(
    model="google/gemma-4-31b-it:free",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)

# 3. بناء الوكيل مع التعليمات (System Prompt)
system_prompt = "أنت مساعد بحث ذكي. ابحث عن الموضوع، ثم استخدم fetch_url لقراءة أول 3 نتائج بتمعن، ثم لخص الإجابة النهائية بناءً على ما قرأت."

agent_executor = create_react_agent(model_nemotron, tools, prompt=system_prompt)

print("✅ تم بناء الوكيل بنجاح! جاهز للاختبار.")

✅ تم بناء الوكيل بنجاح! جاهز للاختبار.


/tmp/ipykernel_3307/1810993675.py:32: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(model_nemotron, tools, prompt=system_prompt)


In [44]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


# استخدام نسخة Llama الأكبر والأذكى
model_final = ChatOpenAI(
    model="meta-llama/llama-3.1-70b-instruct",
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    openai_api_base="https://openrouter.ai/api/v1"
)


system_prompt = "أنت خبير بحث. يجب عليك الآن استخدام أداة internet_search للبحث عن مشاريع سدايا 2025 ثم تلخيصها. لا تعتذر ولا تجب من ذاكرتك."

agent_executor = create_react_agent(model_final, tools, prompt=system_prompt)

print("✅ الوكيل جاهز ومستعد للبحث باستخدام Llama 3.1 (70B)!")

✅ الوكيل جاهز ومستعد للبحث باستخدام Llama 3.1 (70B)!


/tmp/ipykernel_3307/3807870896.py:15: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(model_final, tools, prompt=system_prompt)


In [45]:
import requests
import json

# OpenRouter API base URL
OPENROUTER_API_BASE = "https://openrouter.ai/api/v1"

# Endpoint to list models (this is a common pattern, might need adjustment if OpenRouter uses a different one)
models_url = f"{OPENROUTER_API_BASE}/models"

headers = {
    "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
    "Content-Type": "application/json"
}

print("Attempting to fetch available models from OpenRouter...")

try:
    response = requests.get(models_url, headers=headers)
    response.raise_for_status() # Raise an exception for HTTP errors
    models_data = response.json()

    print("\nAvailable Models on OpenRouter:")
    # Print only the model IDs for clarity
    for model in models_data.get('data', []):
        print(model.get('id'))

except requests.exceptions.RequestException as e:
    print(f"Error fetching models: {e}")
    if response.status_code == 401:
        print("Please ensure your OPENROUTER_API_KEY is correct and active.")
    elif response.status_code == 404:
        print("The models endpoint might be different or unavailable. Please check OpenRouter's API documentation.")
except json.JSONDecodeError:
    print("Error decoding JSON response from OpenRouter.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Attempting to fetch available models from OpenRouter...

Available Models on OpenRouter:
anthropic/claude-opus-4.6-fast
z-ai/glm-5.1
google/gemma-4-26b-a4b-it:free
google/gemma-4-26b-a4b-it
google/gemma-4-31b-it:free
google/gemma-4-31b-it
qwen/qwen3.6-plus
z-ai/glm-5v-turbo
arcee-ai/trinity-large-thinking
x-ai/grok-4.20-multi-agent
x-ai/grok-4.20
google/lyria-3-pro-preview
google/lyria-3-clip-preview
kwaipilot/kat-coder-pro-v2
rekaai/reka-edge
xiaomi/mimo-v2-omni
xiaomi/mimo-v2-pro
minimax/minimax-m2.7
openai/gpt-5.4-nano
openai/gpt-5.4-mini
mistralai/mistral-small-2603
z-ai/glm-5-turbo
nvidia/nemotron-3-super-120b-a12b:free
nvidia/nemotron-3-super-120b-a12b
bytedance-seed/seed-2.0-lite
qwen/qwen3.5-9b
openai/gpt-5.4-pro
openai/gpt-5.4
inception/mercury-2
openai/gpt-5.3-chat
google/gemini-3.1-flash-lite-preview
bytedance-seed/seed-2.0-mini
google/gemini-3.1-flash-image-preview
qwen/qwen3.5-35b-a3b
qwen/qwen3.5-27b
qwen/qwen3.5-122b-a10b
qwen/qwen3.5-flash-02-23
liquid/lfm-2-24b-a2b
goo

In [47]:
# هذي الخلية لتشغيل الوكيل وسؤاله
query = "ما هي أبرز مشاريع سدايا في الذكاء الاصطناعي لعام 2025؟"

print(f"جاري البحث عن: {query}...\n")

# تشغيل الوكيل ورؤية النتائج
# تعديل بسيط لعرض النتائج الخام إذا لم يطبع الموديل نصاً مباشراً
for chunk in agent_executor.stream({"messages": [("user", query)]}):
    print(chunk) # هذا سيطبع كل ما يحدث خلف الكواليس
    print("-" * 30)

جاري البحث عن: ما هي أبرز مشاريع سدايا في الذكاء الاصطناعي لعام 2025؟...

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 422, 'total_tokens': 444, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.0001776, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.0001776, 'upstream_inference_prompt_cost': 0.0001688, 'upstream_inference_completions_cost': 8.8e-06}}, 'model_provider': 'openai', 'model_name': 'meta-llama/llama-3.1-70b-instruct', 'system_fingerprint': None, 'id': 'gen-1776063945-ZtS0MSsvD5z4knjCtah9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d85a9-18c7-76b3-9e74-d989402f2be3-0', tool_calls=[{'name': 'ta